# 14 — Hooks 与 Loop Detection：钩子系统与循环检测

本章实现两个独立但互补的稳定性机制：

- **Hooks 系统**：在 Agent 生命周期的关键节点（工具调用前后、出错时）执行外部 shell 命令
- **Loop Detection**：检测 Agent 是否陷入重复模式（精确重复或周期性循环），并及时中断

两者都是生产级 Agent 必须具备的防护机制，而非可选功能。

In [ ]:
import sys
sys.path.insert(0, "..")

# 验证基础模块可以导入
from src.tool_framework import ToolResult, ToolRegistry, Tool, ToolKind
print("基础模块导入成功")

---

## Part 1：Hooks 系统

### 为什么需要 Hooks

Agent 运行时经常需要在特定时机触发外部动作：

```
用户请求
    ↓
[BEFORE_AGENT]  ← 初始化 venv、设置 PATH
    ↓
Agent 运行中
    ↓
  [BEFORE_TOOL]  ← 记录即将调用的工具
  工具执行
  [AFTER_TOOL]   ← 运行 linter、保存日志
    ↓
[AFTER_AGENT]   ← 清理临时文件、发送完成通知
[ON_ERROR]      ← 发送告警、保存现场
```

Hooks 把这些横切关注点（cross-cutting concerns）从核心 agentic loop 中解耦出来，通过配置驱动而非硬编码。

In [ ]:
import asyncio
import hashlib
import os
import subprocess
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional


class HookTrigger(Enum):
    """Hook 触发时机枚举"""
    BEFORE_AGENT = "before_agent"   # Agent 开始前
    AFTER_AGENT  = "after_agent"    # Agent 结束后
    BEFORE_TOOL  = "before_tool"    # 每次工具调用前
    AFTER_TOOL   = "after_tool"     # 每次工具调用后（无论成功失败）
    ON_ERROR     = "on_error"       # Agent 出错时


@dataclass
class HookConfig:
    """单条 Hook 配置"""
    trigger: HookTrigger
    command: str            # shell 命令，支持 {key} 占位符
    timeout: float = 10.0   # 超时秒数
    enabled: bool = True
    env: dict = field(default_factory=dict)  # 额外环境变量


@dataclass
class HookResult:
    """单条 Hook 执行结果"""
    command: str
    exit_code: int
    stdout: str
    stderr: str
    timed_out: bool

    @property
    def success(self) -> bool:
        return self.exit_code == 0 and not self.timed_out


print("HookTrigger, HookConfig, HookResult 定义完成")
for t in HookTrigger:
    print(f"  {t.name:15s} = {t.value!r}")

### HookSystem 核心实现

关键设计决策：
- 使用 `asyncio.create_subprocess_shell` 实现真异步执行，不阻塞 Agent 主循环
- 通过 `AI_AGENT_*` 环境变量向 hook 脚本传递上下文，而不是命令行参数（避免注入风险）
- `{key}` 占位符仅用于不含敏感数据的简单值（如工具名、文件路径）

In [ ]:
class HookSystem:
    """Hook 系统：管理和执行生命周期钩子"""

    def __init__(self, hooks: list[HookConfig]):
        self.hooks = hooks
        self._session_id = os.urandom(4).hex()  # 简单的会话标识符

    async def trigger(
        self,
        event: HookTrigger,
        context: dict | None = None
    ) -> list[HookResult]:
        """触发指定事件的所有 hook，返回执行结果列表"""
        context = context or {}
        results: list[HookResult] = []

        # 筛选出匹配当前事件且已启用的 hook
        matching = [h for h in self.hooks if h.trigger == event and h.enabled]

        for hook in matching:
            result = await self._run_hook(hook, context)
            results.append(result)

        return results

    async def _run_hook(self, hook: HookConfig, context: dict) -> HookResult:
        """执行单条 hook"""
        # 1. 格式化命令：把 context 中的值填入 {key} 占位符
        try:
            cmd = hook.command.format(**context)
        except KeyError:
            # 占位符找不到对应值时，保持原样
            cmd = hook.command

        # 2. 构造环境变量：当前环境 + AI_AGENT_* 系列 + hook 自定义
        env = os.environ.copy()
        env.update({
            "AI_AGENT_CWD":          os.getcwd(),
            "AI_AGENT_TOOL_NAME":    str(context.get("tool_name", "")),
            "AI_AGENT_TOOL_SUCCESS": str(context.get("tool_success", "")).lower(),
            "AI_AGENT_SESSION_ID":   self._session_id,
        })
        env.update(hook.env)  # hook 自定义变量优先级最高

        # 3. 启动子进程（异步，不阻塞事件循环）
        try:
            proc = await asyncio.create_subprocess_shell(
                cmd,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                env=env,
            )
        except Exception as exc:
            # 进程创建失败（如命令不存在）
            return HookResult(
                command=cmd,
                exit_code=-1,
                stdout="",
                stderr=str(exc),
                timed_out=False,
            )

        # 4. 等待完成，超时则强制终止
        timed_out = False
        try:
            stdout_bytes, stderr_bytes = await asyncio.wait_for(
                proc.communicate(),
                timeout=hook.timeout,
            )
        except asyncio.TimeoutError:
            timed_out = True
            proc.kill()
            stdout_bytes, stderr_bytes = await proc.communicate()

        return HookResult(
            command=cmd,
            exit_code=proc.returncode or 0,
            stdout=stdout_bytes.decode(errors="replace").strip(),
            stderr=stderr_bytes.decode(errors="replace").strip(),
            timed_out=timed_out,
        )

    def add_hook(self, hook: HookConfig) -> None:
        """运行时动态添加 hook"""
        self.hooks.append(hook)

    def disable_hook(self, trigger: HookTrigger) -> int:
        """禁用指定触发器的所有 hook，返回禁用数量"""
        count = 0
        for h in self.hooks:
            if h.trigger == trigger and h.enabled:
                h.enabled = False
                count += 1
        return count


print("HookSystem 定义完成")

### 测试 1：基本 Hook 执行

In [ ]:
async def test_basic_hook():
    """测试：after_tool hook 能正确执行并捕获输出"""
    hooks = [
        HookConfig(
            trigger=HookTrigger.AFTER_TOOL,
            command="echo '工具 {tool_name} 执行完毕，状态：{tool_success}'",
            timeout=5.0,
        ),
        HookConfig(
            trigger=HookTrigger.AFTER_TOOL,
            command="echo 会话ID=$AI_AGENT_SESSION_ID",
            timeout=5.0,
        ),
    ]

    system = HookSystem(hooks)
    context = {"tool_name": "read_file", "tool_success": "true"}
    results = await system.trigger(HookTrigger.AFTER_TOOL, context)

    print(f"执行了 {len(results)} 个 hook")
    for i, r in enumerate(results):
        status = "成功" if r.success else "失败"
        print(f"  Hook {i+1} [{status}]")
        print(f"    命令:   {r.command}")
        print(f"    stdout: {r.stdout}")
        print(f"    stderr: {r.stderr}")
        print(f"    退出码: {r.exit_code}")

await test_basic_hook()

### 测试 2：超时处理

In [ ]:
async def test_timeout_hook():
    """测试：超时的 hook 会被强制终止，不阻塞后续流程"""
    hooks = [
        HookConfig(
            trigger=HookTrigger.ON_ERROR,
            command="sleep 10",  # 故意超时
            timeout=0.5,         # 只等 0.5 秒
        ),
        HookConfig(
            trigger=HookTrigger.ON_ERROR,
            command="echo '告警：Agent 出错'",  # 这条应该正常执行
            timeout=5.0,
        ),
    ]

    system = HookSystem(hooks)
    context = {"error": "工具执行失败"}

    print("触发 ON_ERROR hooks...")
    results = await system.trigger(HookTrigger.ON_ERROR, context)

    for i, r in enumerate(results):
        print(f"Hook {i+1}:")
        print(f"  命令:    {r.command}")
        print(f"  超时:    {r.timed_out}")
        print(f"  成功:    {r.success}")
        print(f"  stdout:  {r.stdout or '（无输出）'}")

await test_timeout_hook()

### 测试 3：日志记录 Hook（写文件场景）

In [ ]:
import tempfile
import os

async def test_logging_hook():
    """测试：hook 将工具调用记录写入日志文件"""
    log_file = tempfile.mktemp(suffix="_tool_log.txt")

    hooks = [
        HookConfig(
            trigger=HookTrigger.AFTER_TOOL,
            # 用环境变量而非占位符，避免引号问题
            command=f"echo "$AI_AGENT_TOOL_NAME success=$AI_AGENT_TOOL_SUCCESS" >> {log_file}",
            timeout=5.0,
        ),
    ]

    system = HookSystem(hooks)

    # 模拟 3 次工具调用
    tool_calls = [
        {"tool_name": "read_file",   "tool_success": "true"},
        {"tool_name": "write_file",  "tool_success": "true"},
        {"tool_name": "web_search",  "tool_success": "false"},
    ]

    for ctx in tool_calls:
        await system.trigger(HookTrigger.AFTER_TOOL, ctx)

    # 读取日志验证
    if os.path.exists(log_file):
        with open(log_file) as f:
            content = f.read()
        print("日志文件内容：")
        print(content)
        os.unlink(log_file)
    else:
        print("日志文件未创建")

await test_logging_hook()

### 测试 4：BEFORE_AGENT / 环境初始化

In [ ]:
async def test_before_agent_hook():
    """测试：before_agent hook 可用于输出环境信息"""
    hooks = [
        HookConfig(
            trigger=HookTrigger.BEFORE_AGENT,
            command="echo CWD=$AI_AGENT_CWD && python3 --version",
            timeout=5.0,
        ),
        # 这条 hook 被禁用，不应执行
        HookConfig(
            trigger=HookTrigger.BEFORE_AGENT,
            command="echo '这条不应出现'",
            timeout=5.0,
            enabled=False,
        ),
    ]

    system = HookSystem(hooks)
    results = await system.trigger(HookTrigger.BEFORE_AGENT)

    print(f"执行了 {len(results)} 个 hook（应为 1 个，另 1 个被禁用）")
    for r in results:
        print(f"stdout: {r.stdout}")
        print(f"成功:   {r.success}")

await test_before_agent_hook()

### 从 TOML 配置加载 Hooks

生产环境中 hook 配置通常来自 `config.toml`，而不是硬编码在代码里。

```toml
[[hooks]]
trigger = "after_tool"
command = "echo 'Tool {tool_name} called' >> .ai_agent/tool_log.txt"
timeout = 5
enabled = true

[[hooks]]
trigger = "on_error"
command = "curl -s -X POST https://hooks.example.com/alert -d 'Agent error'"
timeout = 10
enabled = true
```

In [ ]:
try:
    import tomllib  # Python 3.11+
except ImportError:
    try:
        import tomli as tomllib  # pip install tomli
    except ImportError:
        tomllib = None


def load_hooks_from_toml(toml_str: str) -> list[HookConfig]:
    """
    从 TOML 字符串解析 hook 配置。
    支持 Python 3.11+ 内置 tomllib，降级到 tomli，或纯文本解析。
    """
    if tomllib is None:
        print("警告：未找到 tomllib/tomli，跳过 TOML 解析")
        return []

    data = tomllib.loads(toml_str)
    hooks = []
    for item in data.get("hooks", []):
        try:
            trigger = HookTrigger(item["trigger"])
        except (KeyError, ValueError) as e:
            print(f"  跳过无效 hook 配置：{e}")
            continue

        hooks.append(HookConfig(
            trigger=trigger,
            command=item["command"],
            timeout=float(item.get("timeout", 10.0)),
            enabled=bool(item.get("enabled", True)),
            env=item.get("env", {}),
        ))
    return hooks


# 演示加载
sample_toml = '''
[[hooks]]
trigger = "after_tool"
command = "echo 'tool={tool_name}'"
timeout = 5
enabled = true

[[hooks]]
trigger = "on_error"
command = "echo 'error occurred'"
timeout = 10
enabled = false
'''

if tomllib:
    loaded = load_hooks_from_toml(sample_toml)
    print(f"从 TOML 加载了 {len(loaded)} 条 hook 配置：")
    for h in loaded:
        print(f"  {h.trigger.value:12s} | {h.command!r:40s} | enabled={h.enabled}")
else:
    print("提示：安装 tomli (pip install tomli) 以支持 TOML 配置")

---

## Part 2：Loop Detection

### 问题描述

LLM 有时会陷入循环。常见的两种模式：

```
精确重复：
  step 1: read_file(path="a.py")  ←┐
  step 2: read_file(path="a.py")   |
  step 3: read_file(path="a.py")  ─┘ 触发检测

周期循环：
  step 1: search(q="x")
  step 2: fetch(url="...")
  step 3: search(q="x")    ← 与 step 1 相同
  step 4: fetch(url="...") ← 与 step 2 相同，触发检测
```

检测原理：对每次工具调用计算签名（工具名 + 排序后的参数），维护历史列表，定期检查末尾是否存在重复。

In [ ]:
@dataclass
class LoopDetectionResult:
    """循环检测结果"""
    loop_type: str          # "exact_repeat" 或 "cycle"
    description: str        # 人类可读的描述
    suggestion: str         # 给 LLM 的建议
    cycle_length: int = 0   # 循环模式的周期长度（精确重复时为 0）


class LoopDetector:
    """
    循环检测器：记录工具调用历史，检测精确重复和周期性循环。
    """

    def __init__(self, max_exact_repeats: int = 3, max_cycle_length: int = 4):
        self.max_exact_repeats = max_exact_repeats
        self.max_cycle_length = max_cycle_length
        self.history: list[str] = []  # 存储调用签名的列表
        self._call_details: list[tuple[str, dict]] = []  # 存储原始调用（调试用）

    def _compute_signature(self, tool_name: str, params: dict) -> str:
        """计算工具调用的唯一签名（对参数排序，保证一致性）"""
        # 将参数转为排序后的字符串再做 hash，避免参数顺序影响结果
        canonical = f"{tool_name}::{sorted(params.items())}"
        return hashlib.md5(canonical.encode()).hexdigest()[:12]

    def record(self, tool_name: str, params: dict) -> None:
        """记录一次工具调用"""
        sig = self._compute_signature(tool_name, params)
        self.history.append(sig)
        self._call_details.append((tool_name, params))

    def check(self) -> LoopDetectionResult | None:
        """
        检查是否存在循环。
        先检测精确重复，再检测周期模式，无问题返回 None。
        """
        if not self.history:
            return None

        # --- 精确重复检测 ---
        last_sig = self.history[-1]
        repeat_count = 0
        for sig in reversed(self.history):
            if sig == last_sig:
                repeat_count += 1
            else:
                break

        if repeat_count >= self.max_exact_repeats:
            # 找到对应的工具名，用于描述信息
            last_tool = self._call_details[-1][0] if self._call_details else "unknown"
            return LoopDetectionResult(
                loop_type="exact_repeat",
                description=(
                    f"工具 '{last_tool}' 使用相同参数连续调用了 {repeat_count} 次"
                ),
                suggestion=(
                    "请换一种方式解决问题：尝试不同的工具、不同的参数，"
                    "或者先处理可能导致工具失败的根本原因。"
                ),
            )

        # --- 周期循环检测 ---
        n = len(self.history)
        for cycle_len in range(2, self.max_cycle_length + 1):
            # 需要至少 cycle_len * 2 条历史才能判断
            if n < cycle_len * 2:
                continue

            last_cycle = self.history[-cycle_len:]
            prev_cycle = self.history[-cycle_len * 2 : -cycle_len]

            if last_cycle == prev_cycle:
                # 找到对应工具名序列
                detail_start = -cycle_len * 2
                tools_in_cycle = [
                    t for t, _ in self._call_details[detail_start : detail_start + cycle_len]
                ]
                cycle_desc = " → ".join(tools_in_cycle)
                return LoopDetectionResult(
                    loop_type="cycle",
                    description=(
                        f"检测到长度为 {cycle_len} 的循环模式：{cycle_desc}"
                    ),
                    suggestion=(
                        f"Agent 在重复执行相同的 {cycle_len} 步操作序列。"
                        "请重新评估当前状态，尝试完全不同的方法。"
                    ),
                    cycle_length=cycle_len,
                )

        return None  # 未检测到循环

    def reset(self) -> None:
        """重置历史（通常在成功完成一个子任务后调用）"""
        self.history.clear()
        self._call_details.clear()

    def summary(self) -> str:
        """返回当前历史的简要摘要（调试用）"""
        if not self._call_details:
            return "历史记录为空"
        lines = [f"共 {len(self._call_details)} 次工具调用："]
        for i, (tool, params) in enumerate(self._call_details):
            sig = self.history[i][:6]  # 签名前 6 位
            params_str = str(params)[:40]
            lines.append(f"  [{i+1}] {sig}... {tool}({params_str})")
        return "\n".join(lines)


print("LoopDetector 定义完成")

### 测试 5：精确重复检测

In [ ]:
def test_exact_repeat_detection():
    """测试：连续调用同一工具同样参数 3 次后触发检测"""
    detector = LoopDetector(max_exact_repeats=3, max_cycle_length=4)

    # 前两次：还不到阈值
    detector.record("read_file", {"path": "/tmp/a.py"})
    result = detector.check()
    print(f"调用 1 次后检测结果: {result}")

    detector.record("read_file", {"path": "/tmp/a.py"})
    result = detector.check()
    print(f"调用 2 次后检测结果: {result}")

    # 第三次：触发
    detector.record("read_file", {"path": "/tmp/a.py"})
    result = detector.check()
    print(f"调用 3 次后检测结果:")
    if result:
        print(f"  类型:     {result.loop_type}")
        print(f"  描述:     {result.description}")
        print(f"  建议:     {result.suggestion}")
    else:
        print("  未检测到循环（不符合预期）")

    # 验证参数不同时不触发
    detector.reset()
    detector.record("read_file", {"path": "/tmp/a.py"})
    detector.record("read_file", {"path": "/tmp/b.py"})  # 不同路径
    detector.record("read_file", {"path": "/tmp/a.py"})
    result = detector.check()
    print(f"\n参数不同时（不应触发）: {result}")

test_exact_repeat_detection()

### 测试 6：循环模式检测

In [ ]:
def test_cycle_detection():
    """测试：A→B→A→B 模式触发长度为 2 的循环检测"""
    detector = LoopDetector(max_exact_repeats=3, max_cycle_length=4)

    # 构造 A→B→A→B 序列
    calls = [
        ("web_search", {"query": "python async"}),
        ("web_fetch",  {"url": "https://example.com"}),
        ("web_search", {"query": "python async"}),
        ("web_fetch",  {"url": "https://example.com"}),
    ]

    print("A→B→A→B 循环测试：")
    for i, (tool, params) in enumerate(calls):
        detector.record(tool, params)
        result = detector.check()
        status = f"  → {result.description}" if result else "  → 未检测到"
        print(f"  步骤 {i+1}: {tool}{status}")

    print()

    # 构造 A→B→C→A→B→C 序列（长度 3 的循环）
    detector.reset()
    calls2 = [
        ("list_dir",   {"path": "."}),
        ("read_file",  {"path": "a.py"}),
        ("write_file", {"path": "b.py", "content": "x"}),
        ("list_dir",   {"path": "."}),
        ("read_file",  {"path": "a.py"}),
        ("write_file", {"path": "b.py", "content": "x"}),
    ]

    print("A→B→C→A→B→C 循环测试：")
    for i, (tool, params) in enumerate(calls2):
        detector.record(tool, params)
        result = detector.check()
        status = f"  → [{result.loop_type}] 周期={result.cycle_length}" if result else "  → 未检测到"
        print(f"  步骤 {i+1}: {tool}{status}")

test_cycle_detection()

### 测试 7：签名一致性（参数顺序无关）

In [ ]:
def test_signature_consistency():
    """测试：参数顺序不同但内容相同，签名应相同"""
    detector = LoopDetector(max_exact_repeats=3)

    # 参数字典顺序不同
    params_a = {"path": "/tmp/x", "encoding": "utf-8"}
    params_b = {"encoding": "utf-8", "path": "/tmp/x"}  # 顺序颠倒

    sig_a = detector._compute_signature("read_file", params_a)
    sig_b = detector._compute_signature("read_file", params_b)

    print(f"params_a 签名: {sig_a}")
    print(f"params_b 签名: {sig_b}")
    print(f"签名相同: {sig_a == sig_b}")

    # 参数不同时签名应不同
    params_c = {"path": "/tmp/y", "encoding": "utf-8"}
    sig_c = detector._compute_signature("read_file", params_c)
    print(f"params_c 签名: {sig_c}")
    print(f"a 与 c 不同: {sig_a != sig_c}")

test_signature_consistency()

### 集成到 Agentic Loop

下面展示如何在现有的 agentic loop 中集成 HookSystem 和 LoopDetector。核心逻辑只需在工具调用的处理分支里各加几行。

In [ ]:
async def run_agentic_loop_with_hooks(
    messages: list[dict],
    tool_registry,
    llm_client,
    hook_system: HookSystem | None = None,
    loop_detector: LoopDetector | None = None,
    max_iterations: int = 20,
) -> str:
    """
    带 Hook 系统和循环检测的 agentic loop 骨架。
    展示集成点，而非完整实现。
    """
    # ── BEFORE_AGENT hook ──────────────────────────────────────────
    if hook_system:
        await hook_system.trigger(HookTrigger.BEFORE_AGENT)

    final_response = ""

    for iteration in range(max_iterations):
        # 调用 LLM（省略实际调用，用占位符代替）
        # response, usage = await llm_client.chat_completion(messages, tools=...)

        # 假设这是一次工具调用响应
        tool_name = "example_tool"   # 实际来自 LLM 响应解析
        tool_params = {"key": "val"}  # 实际来自 LLM 响应解析

        # ── BEFORE_TOOL hook ────────────────────────────────────────
        if hook_system:
            await hook_system.trigger(
                HookTrigger.BEFORE_TOOL,
                context={"tool_name": tool_name}
            )

        # ── 循环检测（工具调用前记录）────────────────────────────────
        if loop_detector:
            loop_detector.record(tool_name, tool_params)
            loop_result = loop_detector.check()
            if loop_result:
                # 把检测结果作为 ToolResult.error 返回给 LLM
                error_msg = (
                    f"[循环检测] {loop_result.description}\n"
                    f"建议：{loop_result.suggestion}"
                )
                # 实际实现中：messages.append(ToolResult.error(error_msg).to_tool_message(...))
                print(f"循环检测触发，中断循环：{error_msg}")
                break

        # ── 执行工具（省略）────────────────────────────────────────
        tool_success = True  # 实际来自 tool_registry.invoke()

        # ── AFTER_TOOL hook ─────────────────────────────────────────
        if hook_system:
            await hook_system.trigger(
                HookTrigger.AFTER_TOOL,
                context={
                    "tool_name": tool_name,
                    "tool_success": str(tool_success).lower(),
                }
            )

        # 模拟结束条件（实际应检查 LLM 是否返回 final_response）
        final_response = "任务完成"
        break

    # ── AFTER_AGENT hook ────────────────────────────────────────────
    if hook_system:
        await hook_system.trigger(HookTrigger.AFTER_AGENT)

    return final_response


# 演示骨架函数
async def demo_integration():
    hook_system = HookSystem([
        HookConfig(
            trigger=HookTrigger.BEFORE_AGENT,
            command="echo '[hook] Agent 开始'",
            timeout=3,
        ),
        HookConfig(
            trigger=HookTrigger.AFTER_TOOL,
            command="echo '[hook] 工具 {tool_name} 执行完毕'",
            timeout=3,
        ),
        HookConfig(
            trigger=HookTrigger.AFTER_AGENT,
            command="echo '[hook] Agent 结束'",
            timeout=3,
        ),
    ])
    loop_detector = LoopDetector(max_exact_repeats=3)

    result = await run_agentic_loop_with_hooks(
        messages=[{"role": "user", "content": "hello"}],
        tool_registry=None,
        llm_client=None,
        hook_system=hook_system,
        loop_detector=loop_detector,
    )
    print(f"最终结果: {result}")

await demo_integration()

### 测试 8：完整的循环检测 + 中断流程

In [ ]:
def test_full_loop_detection_workflow():
    """
    模拟完整场景：Agent 不断重试同一失败工具，
    循环检测在第 3 次时介入并提供建议。
    """
    detector = LoopDetector(max_exact_repeats=3, max_cycle_length=4)

    # 模拟工具调用历史（来自 agentic loop）
    simulated_calls = [
        # 第 1 次正常调用
        ("web_search",  {"query": "openai gpt4 docs"}),
        ("web_fetch",   {"url": "https://platform.openai.com/docs"}),
        # 工具失败，LLM 决定重试（错误的选择）
        ("web_fetch",   {"url": "https://platform.openai.com/docs"}),
        ("web_fetch",   {"url": "https://platform.openai.com/docs"}),
    ]

    print("模拟 Agent 工具调用序列：")
    print("-" * 55)

    for step, (tool, params) in enumerate(simulated_calls, 1):
        detector.record(tool, params)
        detection = detector.check()

        params_str = str(params)[:35]
        print(f"步骤 {step}: {tool}({params_str})")

        if detection:
            print(f"  *** 循环检测触发！***")
            print(f"  类型: {detection.loop_type}")
            print(f"  说明: {detection.description}")
            print(f"  建议: {detection.suggestion}")
            print()
            # 实际中这里会把 error 消息注入对话，让 LLM 换策略
            error_for_llm = ToolResult.error(
                f"{detection.description}\n{detection.suggestion}"
            )
            print(f"  注入 LLM 的错误消息:")
            print(f"  {error_for_llm.content[:80]}")
            break
        else:
            print(f"  正常（未检测到循环）")

    print(f"\n历史摘要：\n{detector.summary()}")

test_full_loop_detection_workflow()

---

## 小结

| 模块 | 作用 |
|---|---|
| `HookTrigger` | 枚举 5 个触发时机：`BEFORE_AGENT`, `AFTER_AGENT`, `BEFORE_TOOL`, `AFTER_TOOL`, `ON_ERROR` |
| `HookConfig` | 描述单条 hook：触发时机、shell 命令、超时、启用开关、额外环境变量 |
| `HookResult` | 记录 hook 执行结果：退出码、stdout/stderr、是否超时 |
| `HookSystem` | 管理并异步执行 hook，支持 `{key}` 占位符和 `AI_AGENT_*` 环境变量传递 |
| `LoopDetectionResult` | 描述检测到的循环：类型、描述、给 LLM 的建议 |
| `LoopDetector` | 对工具调用计算签名，检测精确重复（末尾 N 个相同）和周期循环（序列重复） |

下一章：**Memory 系统** — 实现跨会话的持久化记忆，让 Agent 记住用户偏好和历史上下文。

In [ ]:
# 将核心代码保存到 src/hooks_and_loop_detection.py
import os

os.makedirs("../src", exist_ok=True)

code = '''
import asyncio
import hashlib
import os
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional


class HookTrigger(Enum):
    """Hook 触发时机枚举"""
    BEFORE_AGENT = "before_agent"
    AFTER_AGENT  = "after_agent"
    BEFORE_TOOL  = "before_tool"
    AFTER_TOOL   = "after_tool"
    ON_ERROR     = "on_error"


@dataclass
class HookConfig:
    """单条 Hook 配置"""
    trigger: HookTrigger
    command: str
    timeout: float = 10.0
    enabled: bool = True
    env: dict = field(default_factory=dict)


@dataclass
class HookResult:
    """单条 Hook 执行结果"""
    command: str
    exit_code: int
    stdout: str
    stderr: str
    timed_out: bool

    @property
    def success(self) -> bool:
        return self.exit_code == 0 and not self.timed_out


class HookSystem:
    """Hook 系统：管理和执行生命周期钩子"""

    def __init__(self, hooks: list[HookConfig]):
        self.hooks = hooks
        self._session_id = os.urandom(4).hex()

    async def trigger(
        self,
        event: HookTrigger,
        context: dict | None = None
    ) -> list[HookResult]:
        context = context or {}
        results: list[HookResult] = []
        matching = [h for h in self.hooks if h.trigger == event and h.enabled]
        for hook in matching:
            result = await self._run_hook(hook, context)
            results.append(result)
        return results

    async def _run_hook(self, hook: HookConfig, context: dict) -> HookResult:
        try:
            cmd = hook.command.format(**context)
        except KeyError:
            cmd = hook.command

        env = os.environ.copy()
        env.update({
            "AI_AGENT_CWD":          os.getcwd(),
            "AI_AGENT_TOOL_NAME":    str(context.get("tool_name", "")),
            "AI_AGENT_TOOL_SUCCESS": str(context.get("tool_success", "")).lower(),
            "AI_AGENT_SESSION_ID":   self._session_id,
        })
        env.update(hook.env)

        try:
            proc = await asyncio.create_subprocess_shell(
                cmd,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                env=env,
            )
        except Exception as exc:
            return HookResult(command=cmd, exit_code=-1,
                              stdout="", stderr=str(exc), timed_out=False)

        timed_out = False
        try:
            stdout_bytes, stderr_bytes = await asyncio.wait_for(
                proc.communicate(), timeout=hook.timeout
            )
        except asyncio.TimeoutError:
            timed_out = True
            proc.kill()
            stdout_bytes, stderr_bytes = await proc.communicate()

        return HookResult(
            command=cmd,
            exit_code=proc.returncode or 0,
            stdout=stdout_bytes.decode(errors="replace").strip(),
            stderr=stderr_bytes.decode(errors="replace").strip(),
            timed_out=timed_out,
        )

    def add_hook(self, hook: HookConfig) -> None:
        self.hooks.append(hook)

    def disable_hook(self, trigger: HookTrigger) -> int:
        count = 0
        for h in self.hooks:
            if h.trigger == trigger and h.enabled:
                h.enabled = False
                count += 1
        return count


@dataclass
class LoopDetectionResult:
    """循环检测结果"""
    loop_type: str
    description: str
    suggestion: str
    cycle_length: int = 0


class LoopDetector:
    """循环检测器：记录工具调用历史，检测精确重复和周期性循环"""

    def __init__(self, max_exact_repeats: int = 3, max_cycle_length: int = 4):
        self.max_exact_repeats = max_exact_repeats
        self.max_cycle_length = max_cycle_length
        self.history: list[str] = []
        self._call_details: list[tuple[str, dict]] = []

    def _compute_signature(self, tool_name: str, params: dict) -> str:
        canonical = f"{tool_name}::{sorted(params.items())}"
        return hashlib.md5(canonical.encode()).hexdigest()[:12]

    def record(self, tool_name: str, params: dict) -> None:
        sig = self._compute_signature(tool_name, params)
        self.history.append(sig)
        self._call_details.append((tool_name, params))

    def check(self) -> LoopDetectionResult | None:
        if not self.history:
            return None

        # 精确重复检测
        last_sig = self.history[-1]
        repeat_count = 0
        for sig in reversed(self.history):
            if sig == last_sig:
                repeat_count += 1
            else:
                break

        if repeat_count >= self.max_exact_repeats:
            last_tool = self._call_details[-1][0] if self._call_details else "unknown"
            return LoopDetectionResult(
                loop_type="exact_repeat",
                description=f"工具 \'{last_tool}\' 使用相同参数连续调用了 {repeat_count} 次",
                suggestion="请换一种方式解决问题：尝试不同的工具、不同的参数，或者先处理可能导致工具失败的根本原因。",
            )

        # 周期循环检测
        n = len(self.history)
        for cycle_len in range(2, self.max_cycle_length + 1):
            if n < cycle_len * 2:
                continue
            last_cycle = self.history[-cycle_len:]
            prev_cycle = self.history[-cycle_len * 2 : -cycle_len]
            if last_cycle == prev_cycle:
                detail_start = -cycle_len * 2
                tools_in_cycle = [
                    t for t, _ in self._call_details[detail_start : detail_start + cycle_len]
                ]
                cycle_desc = " -> ".join(tools_in_cycle)
                return LoopDetectionResult(
                    loop_type="cycle",
                    description=f"检测到长度为 {cycle_len} 的循环模式：{cycle_desc}",
                    suggestion=f"Agent 在重复执行相同的 {cycle_len} 步操作序列。请重新评估当前状态，尝试完全不同的方法。",
                    cycle_length=cycle_len,
                )

        return None

    def reset(self) -> None:
        self.history.clear()
        self._call_details.clear()

    def summary(self) -> str:
        if not self._call_details:
            return "历史记录为空"
        lines = [f"共 {len(self._call_details)} 次工具调用："]
        for i, (tool, params) in enumerate(self._call_details):
            sig = self.history[i][:6]
            params_str = str(params)[:40]
            lines.append(f"  [{i+1}] {sig}... {tool}({params_str})")
        return "\n".join(lines)
'''

with open("../src/hooks_and_loop_detection.py", "w", encoding="utf-8") as f:
    f.write(code.strip())

print("已保存到 src/hooks_and_loop_detection.py")